## Install

In [1]:
!apt-get install -y poppler-utils
!pip install typhoon-ocr pdf2image beautifulsoup4 pandas

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (213 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 9.2 MB/s eta 0:00:00


## Import / API Key

In [2]:
import os
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from pdf2image import convert_from_path
import typhoon_ocr

os.environ["TYPHOON_API_KEY"] = "sk-t9KG8HcqEVppXtX0vlMK329ZWTAO0gvKP7hvjlEm7ssGjikM"

## Upload Files

PDFs (1 pdf per 1 voting station)

**Page layout:**


- `page[0]` -> constituency
- `page[1]` -> ignore
- `page[2], [3], [4]` -> party_list
- `page[5]` -> ignore

In [3]:
import os, zipfile
from google.colab import files

uploaded = files.upload()

DATA_ROOT = "data"
os.makedirs(DATA_ROOT, exist_ok=True)

for fname, data in uploaded.items():
    if fname.lower().endswith('.zip'):
        zip_path = os.path.join(DATA_ROOT, fname)
        with open(zip_path, 'wb') as f:
            f.write(data)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(DATA_ROOT)
        os.remove(zip_path)
        print(f"Extracted zip: {fname} → {DATA_ROOT}/")
    else:
        # fallback: single PDFs place at data/<filename>
        with open(os.path.join(DATA_ROOT, fname), 'wb') as f:
            f.write(data)

# Create subdistrict_pdf_map: {subdistrict_name: [pdf_path, ...]}
# Refer folder 1st layer of DATA_ROOT as sub-district name
subdistrict_pdf_map = {}
for entry in sorted(os.listdir(DATA_ROOT)):
    entry_path = os.path.join(DATA_ROOT, entry)
    if not os.path.isdir(entry_path):
        continue
    # sub-district name = 1.คูซอด' --> 'คูซอด'
    subdistrict_name = entry.split('.', 1)[-1] if '.' in entry else entry
    pdfs = []
    for root, dirs, files_in in os.walk(entry_path):
        dirs.sort()
        for fn in sorted(files_in):
            if fn.lower().endswith('.pdf'):
                pdfs.append(os.path.join(root, fn))
    if pdfs:
        subdistrict_pdf_map[subdistrict_name] = pdfs

if not subdistrict_pdf_map:
    # fallback: if upload PDF directly without folder structure
    loose_pdfs = sorted([os.path.join(DATA_ROOT, f)
                         for f in os.listdir(DATA_ROOT)
                         if f.lower().endswith('.pdf')])
    if loose_pdfs:
        subdistrict_pdf_map['unknown'] = loose_pdfs

print(f"Subdistricts found: {len(subdistrict_pdf_map)}")
for sd, paths in subdistrict_pdf_map.items():
    print(f"sub-district {sd}: {len(paths)} PDF(s)")


Saving data.zip to data.zip
Extracted zip: data.zip → data/
Subdistricts found: 11
sub-district ชุดที่1: 1 PDF(s)
sub-district ชุดที่10: 1 PDF(s)
sub-district ชุดที่11: 1 PDF(s)
sub-district ชุดที่2: 1 PDF(s)
sub-district ชุดที่3: 1 PDF(s)
sub-district ชุดที่4: 1 PDF(s)
sub-district ชุดที่5: 1 PDF(s)
sub-district ชุดที่6: 1 PDF(s)
sub-district ชุดที่7: 1 PDF(s)
sub-district ชุดที่8: 1 PDF(s)
sub-district ชุดที่9: 1 PDF(s)


## Pre-DataFrame: 57 Parties


In [ ]:
PARTY_NAMES = [
    "ไทยทรัพย์ทวี",
    "เพื่อชาติไทย",
    "ใหม่",
    "มิติใหม่",
    "รวมใจไทย",
    "รวมไทยสร้างชาติ",
    "พลวัต",
    "ประชาธิปไตยใหม่",
    "เพื่อไทย",
    "ทางเลือกใหม่",
    "เศรษฐกิจ",
    "เสรีรวมไทย",
    "รวมพลังประชาชน",
    "ท้องที่ไทย",
    "อนาคตไทย",
    "พลังเพื่อไทย",
    "ไทยชนะ",
    "พลังสังคมใหม่",
    "สังคมประชาธิปไตยไทย",
    "ฟิวชัน",
    "ไทรวมพลัง",
    "ก้าวอิสระ",
    "ปวงชนไทย",
    "วิชช์นใหม่",
    "เพื่อชีวิตใหม่",
    "คลองไทย",
    "ประชาธิปัตย์",
    "ไทยก้าวหน้า",
    "ไทยภักดี",
    "แรงงานสร้างชาติ",
    "ประชากรไทย",
    "ครูไทยเพื่อประชาชน",
    "ประชาชาติ",
    "สร้างอนาคตไทย",
    "รักชาติ",
    "ไทยพร้อม",
    "ภูมิใจไทย",
    "พลังธรรมใหม่",
    "กรีน",
    "ไทยธรรม",
    "แผ่นดินธรรม",
    "กล้าธรรม",
    "พลังประชารัฐ",
    "โอกาสใหม่",
    "เป็นธรรม",
    "ประชาชน",
    "ประชาไทย",
    "ไทยสร้างไทย",
    "ไทยก้าวใหม่",
    "ประชาอาสาชาติ",
    "พร้อม",
    "เครือข่ายชาวนาแห่งประเทศไทย",
    "ไทยพิทักษ์ธรรม",
    "ความหวังใหม่",
    "ไทยรวมไทย",
    "เพื่อบ้านเมือง",
    "พลังไทยรักชาติ",
]

PARTY_INDEX = {name: idx for idx, name in enumerate(PARTY_NAMES)}
assert len(PARTY_NAMES) == 57, f"Expected 57 parties, got {len(PARTY_NAMES)}"
print(f"Pre-DataFrame ready: {len(PARTY_NAMES)} พรรค")

Pre-DataFrame ready: 57 พรรค


## Helper Functions

In [9]:
CONSTITUENCY_PAGES = {0}
PARTY_LIST_PAGES   = {2, 3, 4}
SKIP_PAGES         = {1, 5}

def extract_meta(ocr_text):
    thai_to_arabic = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = ocr_text.translate(thai_to_arabic)

    def grab(pattern):
        m = re.search(pattern, text)
        if not m:
            return None

        val = m.group(1)
        if not val:
            return None

        val = val.strip()
        return int(val) if val.isdigit() else None

    return {
        "eligible": grab(r"ผู้มีสิทธิเลือกตั้งตามบัญชี.*?จำนวน\s*(\d+)"),
        "show_up": grab(r"ผู้มีสิทธิเลือกตั้งที่มาแสดงตน.*?จำนวน\s*(\d*)"),
        "prepared_cards": grab(r"บัตรเลือกตั้งที่ได้รับจัดสรร.*?จำนวน\s*(\d+)"),
        "used_cards": grab(r"บัตรเลือกตั้งที่ใช้.*?จำนวน\s*(\d+)"),
        "good_cards": grab(r"บัตรดี.*?จำนวน\s*(\d+)"),
        "bad_cards": grab(r"บัตรเสีย.*?จำนวน\s*(\d+)"),
        "no_vote_cards": grab(r"ไม่เลือกบัญชีรายชื่อ.*?จำนวน\s*(\d+)"),
        "remaining_cards": grab(r"บัตรเลือกตั้งที่เหลือ.*?จำนวน\s*(\d+)"),
    }


def clean_score(score_text):
    if not score_text:
        return None
    thai_to_arabic = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    score_text = score_text.translate(thai_to_arabic).strip()

    if score_text in ('-', '–', '—', ''):
        return 0
    match = re.search(r"\d+", score_text)
    return int(match.group()) if match else None


def is_summary_row(cols):
    if not cols:
        return True
    first = cols[0].translate(str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")).strip()
    return not re.match(r"^\d+$", first)


def extract_constituency_rows(ocr_text):
    soup = BeautifulSoup(ocr_text, "html.parser")
    rows = []
    for table in soup.find_all("table"):
        for tr in table.find_all("tr")[1:]:
            cols = [td.get_text(strip=True) for td in tr.find_all("td")]
            if len(cols) < 4:
                continue
            if is_summary_row(cols):
                continue
            rows.append({
                "หมายเลข"    : cols[0],
                "ชื่อ-นามสกุล": cols[1],
                "พรรค"       : cols[2],
                "คะแนน"      : clean_score(cols[3]),
            })
    return rows


def extract_party_list_scores(ocr_text):
    soup = BeautifulSoup(ocr_text, "html.parser")
    scores = {}
    for table in soup.find_all("table"):
        for tr in table.find_all("tr")[1:]:
            cols = [td.get_text(strip=True) for td in tr.find_all("td")]
            if len(cols) < 3:
                continue
            if is_summary_row(cols):
                continue
            num_str = cols[0].translate(str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")).strip()
            if not re.match(r"^\d+$", num_str):
                continue
            party_num = int(num_str)
            if party_num < 1 or party_num > 57:
                continue
            score = clean_score(cols[2])
            # นับทุก party ที่อ่าน party_num ได้ ไม่ว่า score จะ None หรือไม่
            # None หมายถึง OCR อ่านไม่ออกจริงๆ (ไม่ใช่แค่ '-') fallback 0
            scores[party_num - 1] = score if score is not None else 0
    return scores

print("Helper functions defined.")

Helper functions defined.


# Pipeline: Party List

Loop over all PDFs and read page[2-4]


In [ ]:
party_unit_rows = []
debug_rows = []

for subdistrict_name, pdf_paths_sd in subdistrict_pdf_map.items():
    # outer loop: all sub-district
    print("\n" + "#"*60)
    print(f" ตำบล: {subdistrict_name}  ({len(pdf_paths_sd)} หน่วย)")
    print("#"*60)

    for pdf_path in pdf_paths_sd:
        # inner loop: all units in sub-district
        pdf_filename = os.path.basename(pdf_path) # e.g. '7.pdf'
        print("\n" + "="*60)
        print(f" PDF: {pdf_path}")
        print("="*60)

        pages = convert_from_path(pdf_path, 300)
        print(f" Total pages: {len(pages)}")

        ocr_results = {}

        for i, page in enumerate(pages):
            if i in SKIP_PAGES:
                print(f"[Page {i}] SKIPPED")
                continue
            if i not in PARTY_LIST_PAGES:
                print(f"[Page {i}] SKIPPED (constituency pipeline)")
                continue

            img_path = f"page_{i}.jpg"
            page.save(img_path, "JPEG")

            try:
                result = typhoon_ocr.ocr_document(img_path)
                ocr_results[i] = result
                print(f"[Page {i}] Done")
            except Exception as e:
                print(f"[Page {i}] Error: {e}")
                ocr_results[i] = ""

        unit_scores = [0] * 57
        pages_read = 0
        meta = {}
        page_entries = {}  # for debug df

        print("\n")
        for page_idx in sorted(ocr_results.keys()):
            ocr_text = ocr_results[page_idx]

            # for debug
            print("="*60)
            print(f"\nRaw content from page {page_idx}: \n")
            print(ocr_text)

            if page_idx == 2:
                meta = extract_meta(ocr_text)

            scores = extract_party_list_scores(ocr_text)
            for idx, score in scores.items():
                unit_scores[idx] += score

            page_entries[page_idx] = len(scores) # for debug df
            print(f"\n[Page {page_idx}] party scores read: {len(scores)} entries\n")
            pages_read += 1

        total_score = sum(unit_scores)

        if pages_read == 3:
            status = "COMPLETE"
        else:
            status = f"only {pages_read}/3 pages read"

        print(f"\nmeta: {meta}")
        print(f"total score: {total_score} | {status}")

        # sub-district name comes from folder name, not OCR
        # unit = A.pdf (e.g. '7.pdf' --> 7)
        try:
            unit_num = int(pdf_filename.split('.')[0])
        except ValueError:
            unit_num = pdf_filename  # fallback if filename is not a number

        row = {
            "source_pdf"      : pdf_path,
            "ตำบล"            : subdistrict_name,
            "หน่วยเลือกตั้ง"  : unit_num,
            "pages_read"      : pages_read,
            **meta
        }
        for idx, name in enumerate(PARTY_NAMES):
            row[name] = unit_scores[idx]

        party_unit_rows.append(row)

        debug_rows.append({
            "sub-district"  : subdistrict_name,
            "unit"          : unit_num,
            "page 2 entries": page_entries.get(2, 0),
            "page 3 entries": page_entries.get(3, 0),
            "page 4 entries": page_entries.get(4, 0),
            "total entries" : sum(page_entries.values()),
        })

print("\n" + "="*60)
print(f" TOTAL units processed: {len(party_unit_rows)}")

Streaming output truncated to the last 5000 lines.
 Total pages: 6
[Page 0] SKIPPED (constituency pipeline)
[Page 1] SKIPPED
[Page 2] Done
[Page 3] Done
[Page 4] Done
[Page 5] SKIPPED



Raw content from page 2: 

ส.ส. ๕/๑๘ (บช)
สีขาว

รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบบัญชีรายชื่อ

ตามที่ได้มีพระราชกฤษฎีกาให้มีการเลือกตั้งสมาชิกสภาผู้แทนราษฎร และคณะกรรมการการเลือกตั้ง ได้กำหนดให้วันที่ 8 เดือน กุมภาพันธ์ พ.ศ. 2569 เป็นวันเลือกตั้ง นั้น
บัดนี้ คณะกรรมการประจำหน่วยเลือกตั้งได้ดำเนินการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบบัญชีรายชื่อของหน่วยเลือกตั้งที่ 12 หมู่ที่ 12 ตำบล/แขวง/เทศบาล ทุ่งสว่าง อำเภอ/เขต วังเหนือ เขตเลือกตั้งที่ 1 จังหวัด ศรีสะเกษ เสร็จสิ้นเป็นที่เรียบร้อยแล้ว ดังนั้น จึงขอรายงานผลการนับคะแนนของหน่วยเลือกตั้งดังกล่าว ดังนี้
๑. จำนวนผู้มีสิทธิเลือกตั้ง
๑.๑ จำนวนผู้มีสิทธิเลือกตั้งตามบัญชีรายชื่อผู้มีสิทธิเลือกตั้ง จำนวน 173 คน (หนึ่งร้อยเจ็ดสิบสาม)
๑.๒ จำนวนผู้มีสิทธิเลือกตั้งที่มาแสดงตน (เฉพาะวันเลือกตั้ง) จำนวน 104 คน (หนึ่งร้อยสี่)
๒. จำนวนบัตรเลือกตั้ง
๒.๑ จำนวนบัตร

### Debug: Row Entries

In [ ]:
df_debug = pd.DataFrame(debug_rows)
df_debug = df_debug.sort_values(["sub-district", "unit"]).reset_index(drop=True)
df_debug

,sub-district,unit,page 2 entries,page 3 entries,page 4 entries,total entries
0,ทุ่งสว่าง,1,10,24,23,57
1,ทุ่งสว่าง,2,10,24,23,57
2,ทุ่งสว่าง,3,10,24,23,57
3,ทุ่งสว่าง,4,10,24,23,57
4,ทุ่งสว่าง,5,10,0,23,33
...,...,...,...,...,...,...
67,เทศบาลตำบลบุสูง,18,10,24,23,57
68,เทศบาลตำบลบุสูง,19,10,24,23,57
69,เทศบาลตำบลบุสูง,20,10,24,23,57
70,เทศบาลตำบลบุสูง,21,10,24,23,57


In [ ]:
df_debug.to_csv("(debug)party_list_entries.csv", index=False, encoding="utf-8-sig")
print("Saved: (debug)party_list_entries.csv")

Saved: (debug)party_list_entries.csv


In [ ]:
from google.colab import files
files.download("(debug)party_list_entries.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## party_list.csv (grouped by units and by sub-district)

columns: `sub_district`, `party1`, `party2`, ..., `party57`

In [ ]:
df_units = pd.DataFrame(party_unit_rows) ## อันนี้แยกตามหน่วย น่าจะ validate ง่ายกว่า
print("=== party_list (grouped by ) ===")

df_units = df_units.sort_values(
    by=["ตำบล", "หน่วยเลือกตั้ง"],
    ascending=[True, True]
).reset_index(drop=True)

print(f"Rows: {len(df_units)}")

=== party_list (grouped by ) ===
Rows: 72


In [ ]:
df_units

,source_pdf,ตำบล,หน่วยเลือกตั้ง,pages_read,eligible,show_up,prepared_cards,used_cards,good_cards,bad_cards,...,ไทยสร้างไทย,ไทยก้าวใหม่,ประชาอาสาชาติ,พร้อม,เครือข่ายชาวนาแห่งประเทศไทย,ไทยพิทักษ์ธรรม,ความหวังใหม่,ไทยรวมไทย,เพื่อบ้านเมือง,พลังไทยรักชาติ
0,data/1.ทุ่งสว่าง/หน่วยเลือกตั้งที่1/1.pdf,ทุ่งสว่าง,1,3,219,NaN,220,150,NaN,NaN,...,1,1,0,1,0,0,0,0,0,0
1,data/1.ทุ่งสว่าง/หน่วยเลือกตั้งที่2/2.pdf,ทุ่งสว่าง,2,3,310,178.0,300,178,170.0,8.0,...,0,0,0,0,0,0,0,0,0,0
2,data/1.ทุ่งสว่าง/หน่วยเลือกตั้งที่3/3.pdf,ทุ่งสว่าง,3,3,205,125.0,200,125,124.0,0.0,...,1,0,0,0,0,0,0,0,0,0
3,data/1.ทุ่งสว่าง/หน่วยเลือกตั้งที่4/4.pdf,ทุ่งสว่าง,4,3,158,106.0,160,106,104.0,2.0,...,2,0,0,0,0,0,0,0,0,0
4,data/1.ทุ่งสว่าง/หน่วยเลือกตั้งที่5/5.pdf,ทุ่งสว่าง,5,3,318,200.0,320,200,NaN,188.0,...,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,data/4.เทศบาลตำบลบุสูง/หน่วยเลือกตั้งที่18/18.pdf,เทศบาลตำบลบุสูง,18,3,171,NaN,180,128,115.0,8.0,...,0,0,0,0,0,0,0,0,0,0
68,data/4.เทศบาลตำบลบุสูง/หน่วยเลือกตั้งที่19/19.pdf,เทศบาลตำบลบุสูง,19,3,305,NaN,300,206,NaN,NaN,...,0,0,1,0,0,0,0,0,0,1
69,data/4.เทศบาลตำบลบุสูง/หน่วยเลือกตั้งที่20/20.pdf,เทศบาลตำบลบุสูง,20,3,167,80.0,163,80,75.0,5.0,...,0,0,1,0,0,0,0,0,0,0
70,data/4.เทศบาลตำบลบุสูง/หน่วยเลือกตั้งที่21/21.pdf,เทศบาลตำบลบุสูง,21,3,148,91.0,160,91,55.0,5.0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df_units[["ประชาชน", "ภูมิใจไทย"]]

,ประชาชน,ภูมิใจไทย
0,32,26
1,80,38
2,17,68
3,9,35
4,36,57
...,...,...
67,12,27
68,17,68
69,0,30
70,16,34


In [ ]:
df_units["ภูมิใจไทย"].sum()

np.int64(23522)

In [ ]:
df_units[["used_cards","good_cards","bad_cards","no_vote_cards"]]

,used_cards,good_cards,bad_cards,no_vote_cards
0,150,NaN,NaN,3.0
1,178,170.0,8.0,NaN
2,125,124.0,0.0,1.0
3,106,104.0,2.0,NaN
4,200,NaN,188.0,2.0
...,...,...,...,...
67,128,115.0,8.0,5.0
68,206,NaN,NaN,146.0
69,80,75.0,5.0,0.0
70,91,55.0,5.0,1.0


In [ ]:
subdistrict_col = "ตำบล"

cols_to_sum = ["used_cards","good_cards","bad_cards","no_vote_cards"] + PARTY_NAMES

df_agg = (
    df_units
    .groupby(subdistrict_col, dropna=False)[cols_to_sum]
    .sum()
    .reset_index()
    .rename(columns={subdistrict_col: "sub_district"})
)

print("=== party_list (grouped by sub-district) ===") # อันนี้ คือ แยกตามตำบลนะ
print(f"Rows: {len(df_agg)}")

df_agg

=== party_list (grouped by sub-district) ===
Rows: 4


,sub_district,used_cards,good_cards,bad_cards,no_vote_cards,ไทยทรัพย์ทวี,เพื่อชาติไทย,ใหม่,มิติใหม่,รวมใจไทย,...,ไทยสร้างไทย,ไทยก้าวใหม่,ประชาอาสาชาติ,พร้อม,เครือข่ายชาวนาแห่งประเทศไทย,ไทยพิทักษ์ธรรม,ความหวังใหม่,ไทยรวมไทย,เพื่อบ้านเมือง,พลังไทยรักชาติ
0,ทุ่งสว่าง,2239,1686.0,264.0,20.0,117,43,36,10,46,...,7,3,1,2,0,1,1,0,0,4
1,ธาตุ,3654,3040.0,356.0,16.0,275,54,61,6,636,...,19,5,5,3,4,4,4,4,4,10
2,บ่อแก้ว,3134,2668.0,391.0,23.0,89,372,269,118,172,...,6,6,3,4,8,3,3,5,4,248
3,เทศบาลตำบลบุสูง,4992,4385.0,228.0,219.0,18,56,260,9,121,...,16,1,2,0,6,0,2,1,2,4


## Export & Download

In [ ]:
df_agg.to_csv("party_list.csv", index=False, encoding="utf-8-sig")
df_units.to_csv("party_list_by_unit.csv", index=False, encoding="utf-8-sig")

print("Saved: party_list.csv (ระดับตำบล)")
print("Saved: party_list_by_unit.csv (ระดับหน่วย)")

Saved: party_list.csv (ระดับตำบล)
Saved: party_list_by_unit.csv (ระดับหน่วย)


In [ ]:
from google.colab import files
files.download("party_list.csv")
files.download("party_list_by_unit.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Pipeline: Constituency

## Pre-DataFrame: 9 Candidates


In [ ]:
CONSTITUENCY_CANDIDATES = {
    1: "นายสุเกษม เรืองนุช",
    2: "นายเนียม ผลบุญ",
    3: "นายธเนส เครือรัตน์",
    4: "นายวีรชัย จันทร์ดวงศรี",
    5: "นายสิริพงศ์ อังคสกุลเกียรติ",
    6: "นายสุรเดช นาจำปา",
    7: "นายคณีศร คำโสภา",
    8: "นายกุลธวัช สาลี",
    9: "นางสาวฉวี ปิ่นทอม",
}

CAND_COLS = {num: f"ชื่อนามสกุลผู้สมัครหมายเลข{num}" for num in range(1, 10)}

print(f"Constituency candidates pre-loaded: {len(CONSTITUENCY_CANDIDATES)} คน")
for num, name in CONSTITUENCY_CANDIDATES.items():
    print(f"  {num}. {name}")

Constituency candidates pre-loaded: 9 คน
  1. นายสุเกษม เรืองนุช
  2. นายเนียม ผลบุญ
  3. นายธเนส เครือรัตน์
  4. นายวีรชัย จันทร์ดวงศรี
  5. นายสิริพงศ์ อังคสกุลเกียรติ
  6. นายสุรเดช นาจำปา
  7. นายคณีศร คำโสภา
  8. นายกุลธวัช สาลี
  9. นางสาวฉวี ปิ่นทอม


## Helper Functions: Constituency

In [ ]:
def extract_constituency_scores_page0(ocr_text):
    soup = BeautifulSoup(ocr_text, "html.parser")
    scores = {}
    for table in soup.find_all("table"):
        for tr in table.find_all("tr")[1:]:
            cols = [td.get_text(strip=True) for td in tr.find_all("td")]
            if len(cols) < 2:
                continue
            if is_summary_row(cols):
                continue
            num_str = cols[0].translate(str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")).strip()
            if not re.match(r"^\d+$", num_str):
                continue
            cand_num = int(num_str)
            if cand_num < 1 or cand_num > 9:
                continue
            score = clean_score(cols[-1])
            scores[cand_num] = score if score is not None else 0
    return scores

def extract_meta_constituency(ocr_text):
    return extract_meta(ocr_text)

## Pipeline: OCR

In [ ]:
con_unit_rows  = []   # df_units_constituency
con_debug_rows = []   # debug_df_constituency

for subdistrict_name, pdf_paths_sd in subdistrict_pdf_map.items():
    print("\n" + "#"*60)
    print(f" ตำบล: {subdistrict_name}  ({len(pdf_paths_sd)} หน่วย)")
    print("#"*60)

    for pdf_path in pdf_paths_sd:
        pdf_filename = os.path.basename(pdf_path)
        print("\n" + "="*60)
        print(f" PDF: {pdf_path}")
        print("="*60)

        pages = convert_from_path(pdf_path, 300)
        print(f" Total pages: {len(pages)}")

        # อ่านแค่ page[0]
        img_path = "con_page_0.jpg"
        pages[0].save(img_path, "JPEG")

        try:
            ocr_text = typhoon_ocr.ocr_document(img_path)
            print("[Page 0] Done")
        except Exception as e:
            print(f"[Page 0] Error: {e}")
            ocr_text = ""

        # debug: raw output
        print("="*60)
        print("\nRaw content from page 0:\n")
        print(ocr_text)
        print("="*60)

        meta   = extract_meta_constituency(ocr_text)
        scores = extract_constituency_scores_page0(ocr_text)

        print(f"\n[Page 0] constituency scores read: {len(scores)} entries")
        print(f"scores: {scores}")

        try:
            unit_num = int(pdf_filename.split('.')[0])
        except ValueError:
            unit_num = pdf_filename

        row = {
            "source_pdf"          : pdf_path,
            "ตำบลหน่วยเลือกตั้ง" : subdistrict_name,
            "eligible"            : meta.get("eligible"),
            "show_up"             : meta.get("show_up"),
            "prepared_cards"      : meta.get("prepared_cards"),
            "used_cards"          : meta.get("used_cards"),
            "good_cards"          : meta.get("good_cards"),
            "bad_cards"           : meta.get("bad_cards"),
        }
        for num, col_name in CAND_COLS.items():
            row[col_name] = scores.get(num, 0)

        con_unit_rows.append(row)

        # debug row
        con_debug_rows.append({
            "sub-district"  : subdistrict_name,
            "unit"          : unit_num,
            "page 0 entries": len(scores),
            "completeness"    : "yes" if len(scores) == 9 else f"{len(scores)}/9",
        })

print("\n" + "="*60)
print(f" TOTAL units processed: {len(con_unit_rows)}")


############################################################
 ตำบล: ทุ่งสว่าง  (15 หน่วย)
############################################################

 PDF: data/1.ทุ่งสว่าง/หน่วยเลือกตั้งที่1/1.pdf
 Total pages: 6
[Page 0] Done

Raw content from page 0:

ส.ส. ๕/๑๘
สีขาว

ชุดที่ ๑ เก็บไว้ในเล่มเพื่อส่งคณะกรรมการการเลือกตั้งประจำเขตเลือกตั้ง (แบบแบ่งเขตเลือกตั้ง)

รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง

ตามที่ได้มีพระราชกฤษฎีกาให้มีการเลือกตั้งสมาชิกสภาผู้แทนราษฎร และคณะกรรมการการเลือกตั้ง ได้กำหนดให้วันที่ 28 เดือน กุมภาพันธ์ พ.ศ. 2567 เป็นวันเลือกตั้ง นั้น
บัดนี้ คณะกรรมการประจำหน่วยเลือกตั้งได้ดำเนินการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขต เลือกตั้งของหน่วยเลือกตั้งที่ 1 หมู่ที่ 1 ตำบล/แขวง/เทศบาล กะท่าง อำเภอ/เขต วังนิ่ม เขตเลือกตั้งที่ 1 จังหวัด ศรีสะเกษ เสร็จสิ้นเป็นที่เรียบร้อยแล้ว ดังนั้น จึงขอรายงานผลการนับคะแนนของ หน่วยเลือกตั้งดังกล่าว ดังนี้
๑. จำนวนผู้มีสิทธิเลือกตั้ง
    ๑.๑ จำนวนผู้มีสิทธิเลือกตั้งตามบัญชีรายชื่อผู้มีสิทธิเลือกตั้ง จำนวน 219 คน (สอง

## Export Dataframe

In [ ]:
df_units_constituency = pd.DataFrame(con_unit_rows)

rename_map = {f"ชื่อนามสกุลผู้สมัครหมายเลข{num}": CONSTITUENCY_CANDIDATES[num]
              for num in range(1, 10)}
df_units_display = df_units_constituency.rename(columns=rename_map)

print("\n=== df_units_constituency ===")
print(f"Rows: {len(df_units_constituency)}")
display(df_units_display)


=== df_units_constituency ===
Rows: 72


,source_pdf,ตำบลหน่วยเลือกตั้ง,eligible,show_up,prepared_cards,used_cards,good_cards,bad_cards,นายสุเกษม เรืองนุช,นายเนียม ผลบุญ,นายธเนส เครือรัตน์,นายวีรชัย จันทร์ดวงศรี,นายสิริพงศ์ อังคสกุลเกียรติ,นายสุรเดช นาจำปา,นายคณีศร คำโสภา,นายกุลธวัช สาลี,นางสาวฉวี ปิ่นทอม
0,data/1.ทุ่งสว่าง/หน่วยเลือกตั้งที่1/1.pdf,ทุ่งสว่าง,219,150.0,220,150,147.0,2.0,0,0,0,0,0,0,0,0,0
1,data/1.ทุ่งสว่าง/หน่วยเลือกตั้งที่10/10.pdf,ทุ่งสว่าง,236,129.0,240,129,124.0,5.0,10,2,44,0,64,0,23,0,591
2,data/1.ทุ่งสว่าง/หน่วยเลือกตั้งที่11/11.pdf,ทุ่งสว่าง,240,171.0,240,171,177.0,4.0,13,1,71,0,80,0,1,1,0
3,data/1.ทุ่งสว่าง/หน่วยเลือกตั้งที่12/12.pdf,ทุ่งสว่าง,173,104.0,180,104,NaN,NaN,4,0,54,0,41,0,0,0,0
4,data/1.ทุ่งสว่าง/หน่วยเลือกตั้งที่13/13.pdf,ทุ่งสว่าง,222,125.0,220,125,120.0,4.0,5,0,28,1,35,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,data/4.เทศบาลตำบลบุสูง/หน่วยเลือกตั้งที่5/5.pdf,เทศบาลตำบลบุสูง,710,427.0,680,427,NaN,16.0,39,3,137,0,218,2,0,0,11
68,data/4.เทศบาลตำบลบุสูง/หน่วยเลือกตั้งที่6/6.pdf,เทศบาลตำบลบุสูง,373,195.0,360,195,190.0,3.0,12,0,89,1,84,0,2,0,2
69,data/4.เทศบาลตำบลบุสูง/หน่วยเลือกตั้งที่7/7.pdf,เทศบาลตำบลบุสูง,361,296.0,360,296,228.0,6.0,12,0,100,0,110,1,0,0,5
70,data/4.เทศบาลตำบลบุสูง/หน่วยเลือกตั้งที่8/8.pdf,เทศบาลตำบลบุสูง,666,357.0,640,357,350.0,6.0,11,1,134,2,193,0,0,0,9


In [ ]:
debug_df_constituency = pd.DataFrame(con_debug_rows)
print("\n=== debug_df_constituency ===")
display(debug_df_constituency)


=== debug_df_constituency ===


,sub-district,unit,page 0 entries,completeness
0,ทุ่งสว่าง,1,9,yes
1,ทุ่งสว่าง,10,9,yes
2,ทุ่งสว่าง,11,9,yes
3,ทุ่งสว่าง,12,8,8/9
4,ทุ่งสว่าง,13,9,yes
...,...,...,...,...
67,เทศบาลตำบลบุสูง,5,9,yes
68,เทศบาลตำบลบุสูง,6,9,yes
69,เทศบาลตำบลบุสูง,7,9,yes
70,เทศบาลตำบลบุสูง,8,9,yes


In [ ]:
score_cols = list(CAND_COLS.values())

agg_dict = {"source_pdf": "count"}
agg_dict.update({col: "sum" for col in score_cols})

df_agg_constituency = (
    df_units_constituency
    .groupby("ตำบลหน่วยเลือกตั้ง", as_index=False)
    .agg(agg_dict)
    .rename(columns={"source_pdf": "จำนวนหน่วย"})
)

df_agg_constituency = df_agg_constituency.rename(columns=rename_map)

print("\n=== df_agg_constituency ===")
print(f"Rows: {len(df_agg_constituency)}")
display(df_agg_constituency)


=== df_agg_constituency ===
Rows: 4


,ตำบลหน่วยเลือกตั้ง,จำนวนหน่วย,นายสุเกษม เรืองนุช,นายเนียม ผลบุญ,นายธเนส เครือรัตน์,นายวีรชัย จันทร์ดวงศรี,นายสิริพงศ์ อังคสกุลเกียรติ,นายสุรเดช นาจำปา,นายคณีศร คำโสภา,นายกุลธวัช สาลี,นางสาวฉวี ปิ่นทอม
0,ทุ่งสว่าง,15,138,11,751,12,1064,2,29,1,610
1,ธาตุ,16,185,528,1444,7,1902,5,10,4,50
2,บ่อแก้ว,19,129,17,1011,14,1601,2,8,0,23
3,เทศบาลตำบลบุสูง,22,255,15,1874,19,2448,36,14,6,124


In [ ]:
df_units_display.to_csv("constituency_units.csv",      index=False, encoding="utf-8-sig")
df_agg_constituency.to_csv("constituency_agg.csv",     index=False, encoding="utf-8-sig")
debug_df_constituency.to_csv("constituency_debug.csv", index=False, encoding="utf-8-sig")

print("\nSaved: constituency_units.csv")
print("Saved: constituency_agg.csv")
print("Saved: constituency_debug.csv")

files.download("constituency_units.csv")
files.download("constituency_agg.csv")
files.download("constituency_debug.csv")


Saved: constituency_units.csv
Saved: constituency_agg.csv
Saved: constituency_debug.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Pipeline: ล่วงหน้านอกเขต

## Pre-Dataframe

In [14]:
CONSTITUENCY_CANDIDATES = {
    1: "นายสุเกษม เรืองนุช",
    2: "นายเนียม ผลบุญ",
    3: "นายธเนส เครือรัตน์",
    4: "นายวีรชัย จันทร์ดวงศรี",
    5: "นายสิริพงศ์ อังคสกุลเกียรติ",
    6: "นายสุรเดช นาจำปา",
    7: "นายคณีศร คำโสภา",
    8: "นายกุลธวัช สาลี",
    9: "นางสาวฉวี ปิ่นทอม",
}

CAND_COLS_EA = {num: f"ชื่อนามสกุลผู้สมัครหมายเลข{num}" for num in range(1, 10)}

print(f"Constituency candidates pre-loaded: {len(CONSTITUENCY_CANDIDATES)} คน")
for num, name in CONSTITUENCY_CANDIDATES.items():
    print(f"  {num}. {name}")

Constituency candidates pre-loaded: 9 คน
  1. นายสุเกษม เรืองนุช
  2. นายเนียม ผลบุญ
  3. นายธเนส เครือรัตน์
  4. นายวีรชัย จันทร์ดวงศรี
  5. นายสิริพงศ์ อังคสกุลเกียรติ
  6. นายสุรเดช นาจำปา
  7. นายคณีศร คำโสภา
  8. นายกุลธวัช สาลี
  9. นางสาวฉวี ปิ่นทอม


In [15]:
PARTY_NAMES_EA = [
    "ไทยทรัพย์ทวี",
    "เพื่อชาติไทย",
    "ใหม่",
    "มิติใหม่",
    "รวมใจไทย",
    "รวมไทยสร้างชาติ",
    "พลวัต",
    "ประชาธิปไตยใหม่",
    "เพื่อไทย",
    "ทางเลือกใหม่",
    "เศรษฐกิจ",
    "เสรีรวมไทย",
    "รวมพลังประชาชน",
    "ท้องที่ไทย",
    "อนาคตไทย",
    "พลังเพื่อไทย",
    "ไทยชนะ",
    "พลังสังคมใหม่",
    "สังคมประชาธิปไตยไทย",
    "ฟิวชัน",
    "ไทรวมพลัง",
    "ก้าวอิสระ",
    "ปวงชนไทย",
    "วิชช์นใหม่",
    "เพื่อชีวิตใหม่",
    "คลองไทย",
    "ประชาธิปัตย์",
    "ไทยก้าวหน้า",
    "ไทยภักดี",
    "แรงงานสร้างชาติ",
    "ประชากรไทย",
    "ครูไทยเพื่อประชาชน",
    "ประชาชาติ",
    "สร้างอนาคตไทย",
    "รักชาติ",
    "ไทยพร้อม",
    "ภูมิใจไทย",
    "พลังธรรมใหม่",
    "กรีน",
    "ไทยธรรม",
    "แผ่นดินธรรม",
    "กล้าธรรม",
    "พลังประชารัฐ",
    "โอกาสใหม่",
    "เป็นธรรม",
    "ประชาชน",
    "ประชาไทย",
    "ไทยสร้างไทย",
    "ไทยก้าวใหม่",
    "ประชาอาสาชาติ",
    "พร้อม",
    "เครือข่ายชาวนาแห่งประเทศไทย",
    "ไทยพิทักษ์ธรรม",
    "ความหวังใหม่",
    "ไทยรวมไทย",
    "เพื่อบ้านเมือง",
    "พลังไทยรักชาติ",
]

PARTY_INDEX_EA = {name: idx for idx, name in enumerate(PARTY_NAMES_EA)}
assert len(PARTY_NAMES_EA) == 57, f"Expected 57 parties, got {len(PARTY_NAMES_EA)}"
print(f"Pre-DataFrame ready: {len(PARTY_NAMES_EA)} พรรค")

Pre-DataFrame ready: 57 พรรค


## File Structure - Organization

In [16]:
import re as _re

early_voting_pdf_list = []

for entry in sorted(os.listdir(DATA_ROOT),
                    key=lambda x: int(_re.search(r'\d+', x).group()) if _re.search(r'\d+', x) else 0):
    entry_path = os.path.join(DATA_ROOT, entry)
    if not os.path.isdir(entry_path):
        continue
    if not entry.startswith('ชุดที่'):
        continue
    for fn in sorted(os.listdir(entry_path)):
        if fn.lower().endswith('.pdf'):
            early_voting_pdf_list.append((entry, os.path.join(entry_path, fn)))

print(f"Early-voting PDF sets found: {len(early_voting_pdf_list)}")
for label, path in early_voting_pdf_list:
    print(f"  {label}: {path}")

Early-voting PDF sets found: 11
  ชุดที่1: data/ชุดที่1/1.pdf
  ชุดที่2: data/ชุดที่2/2.pdf
  ชุดที่3: data/ชุดที่3/3.pdf
  ชุดที่4: data/ชุดที่4/4.pdf
  ชุดที่5: data/ชุดที่5/5.pdf
  ชุดที่6: data/ชุดที่6/6.pdf
  ชุดที่7: data/ชุดที่7/7.pdf
  ชุดที่8: data/ชุดที่8/8.pdf
  ชุดที่9: data/ชุดที่9/9.pdf
  ชุดที่10: data/ชุดที่10/10.pdf
  ชุดที่11: data/ชุดที่11/11.pdf


## Helper Function

In [ ]:
def extract_meta_ea_constituency(ocr_text):
    thai_to_arabic = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = ocr_text.translate(thai_to_arabic)

    def grab(pattern):
        m = re.search(pattern, text)
        if not m:
            return None
        val = m.group(1).strip()
        return int(val) if val.isdigit() else None

    return {
        "received_cards"  : grab(r"ได้รับบัตรเลือกตั้ง.*?แบ่งเขต.*?ทั้งหมด\s*(\d+)"),
        "good_cards"      : grab(r"2\.1\s*บัตรดี\s*(\d+)"),
        "bad_cards"       : grab(r"2\.2\s*บัตรเสีย\s*(\d+)"),
        "no_vote_cards"   : grab(r"2\.3.*?ไม่เลือกผู้สมัครผู้ใด\s*(\d+)"),
    }


def extract_meta_ea_partylist(ocr_text):
    thai_to_arabic = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = ocr_text.translate(thai_to_arabic)

    def grab(pattern):
        m = re.search(pattern, text)
        if not m:
            return None
        val = m.group(1).strip()
        return int(val) if val.isdigit() else None

    return {
        "received_cards"  : grab(r"ได้รับบัตรเลือกตั้ง.*?บัญชีรายชื่อ.*?ทั้งหมด\s*(\d+)"),
        "good_cards"      : grab(r"2\.1\s*บัตรดี\s*(\d+)"),
        "bad_cards"       : grab(r"2\.2\s*บัตรเสีย\s*(\d+)"),
        "no_vote_cards"   : grab(r"2\.3.*?ไม่เลือกบัญชีรายชื่อ.*?ใด\s*(\d+)"),
    }


def extract_ea_constituency_scores(ocr_text):
    soup = BeautifulSoup(ocr_text, "html.parser")
    scores = {}
    for table in soup.find_all("table"):
        for tr in table.find_all("tr")[1:]:  # skip header
            cols = [td.get_text(strip=True) for td in tr.find_all("td")]
            if len(cols) < 2:
                continue

            num_str = cols[0].translate(str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")).strip()
            if not re.match(r"^\d+$", num_str):
                continue
            cand_num = int(num_str)
            if cand_num < 1 or cand_num > 9:
                continue

            score = clean_score(cols[-1])
            scores[cand_num] = score if score is not None else 0
    return scores


def extract_ea_partylist_scores(ocr_text):
    soup = BeautifulSoup(ocr_text, "html.parser")
    scores = {}
    for table in soup.find_all("table"):
        for tr in table.find_all("tr")[1:]:
            cols = [td.get_text(strip=True) for td in tr.find_all("td")]
            if len(cols) < 3:
                continue
            if is_summary_row(cols):
                continue
            num_str = cols[0].translate(str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")).strip()
            if not re.match(r"^\d+$", num_str):
                continue
            party_num = int(num_str)
            if party_num < 1 or party_num > 57:
                continue
            score = clean_score(cols[2])
            scores[party_num - 1] = score if score is not None else 0  # 0-indexed
    return scores

## OCR

In [ ]:
EA_CONSTITUENCY_PAGES = {0, 1}   
EA_PARTYLIST_PAGES    = {2, 3, 4, 5} 

ea_con_rows    = []  
ea_party_rows  = []   
ea_debug_rows  = []   # debug

for set_label, pdf_path in early_voting_pdf_list:
    pdf_filename = os.path.basename(pdf_path)  # e.g. '3.pdf'

    print("\n" + "="*60)
    print(f" ชุด: {set_label}  |  PDF: {pdf_path}")
    print("="*60)

    pages = convert_from_path(pdf_path, 300)
    print(f" Total pages: {len(pages)}")

    ocr_results = {}

    for i, page in enumerate(pages):
        img_path = f"ea_page_{i}.jpg"
        page.save(img_path, "JPEG")
        try:
            result = typhoon_ocr.ocr_document(img_path)
            ocr_results[i] = result
            print(f"[Page {i}] Done")
        except Exception as e:
            print(f"[Page {i}] Error: {e}")
            ocr_results[i] = ""

    con_scores  = {}   
    con_meta    = {}
    con_pages_read = 0

    for pi in sorted(EA_CONSTITUENCY_PAGES):
        if pi not in ocr_results:
            continue
        ocr_text = ocr_results[pi]

        print("\n" + "-"*50)
        print(f"[Page {pi}] Raw (สส เขต):")
        print(ocr_text)

        if pi == 0:
            con_meta = extract_meta_ea_constituency(ocr_text)

        page_scores = extract_ea_constituency_scores(ocr_text)

        for k, v in page_scores.items():
            con_scores[k] = con_scores.get(k, 0) + v

        print(f"[Page {pi}] constituency scores read: {len(page_scores)} entries | {page_scores}")
        con_pages_read += 1

    con_row = {
        "source_pdf"   : pdf_path,
        "ชุด"          : set_label,
        **con_meta,
    }
    for num, col_name in CAND_COLS_EA.items():
        con_row[col_name] = con_scores.get(num, 0)
    ea_con_rows.append(con_row)

    party_scores = [0] * 57
    party_meta   = {}
    party_pages_read = 0

    for pi in sorted(EA_PARTYLIST_PAGES):
        if pi not in ocr_results:
            continue
        ocr_text = ocr_results[pi]

        print("\n" + "-"*50)
        print(f"[Page {pi}] Raw (บัญชีรายชื่อ):")
        print(ocr_text)

        if pi == 2:
            party_meta = extract_meta_ea_partylist(ocr_text)

        p_scores = extract_ea_partylist_scores(ocr_text)
        for idx, score in p_scores.items():
            party_scores[idx] += score

        print(f"[Page {pi}] party scores read: {len(p_scores)} entries")
        party_pages_read += 1

    total_party_score = sum(party_scores)
    party_status = "COMPLETE" if party_pages_read == 4 else f"only {party_pages_read}/4 pages read"

    print(f"\ncon_meta   : {con_meta}")
    print(f"party_meta : {party_meta}")
    print(f"total party score: {total_party_score} | {party_status}")

    party_row = {
        "source_pdf"   : pdf_path,
        "ชุด"          : set_label,
        **party_meta,
    }
    for idx, name in enumerate(PARTY_NAMES_EA):
        party_row[name] = party_scores[idx]
    ea_party_rows.append(party_row)

    # debug row
    ea_debug_rows.append({
        "ชุด"               : set_label,
        "source_pdf"        : pdf_path,
        "con_pages_read"    : con_pages_read,
        "con_scores_count"  : len(con_scores),
        "con_completeness"  : "yes" if len(con_scores) == 9 else f"{len(con_scores)}/9",
        "party_pages_read"  : party_pages_read,
        "party_total_score" : total_party_score,
        "party_completeness": party_status,
    })

print("\n" + "="*60)
print(f" TOTAL ชุด processed: {len(ea_con_rows)}")


 ชุด: ชุดที่1  |  PDF: data/ชุดที่1/1.pdf
 Total pages: 6
[Page 0] Done
[Page 1] Done
[Page 2] Done
[Page 3] Done
[Page 4] Done
[Page 5] Done

--------------------------------------------------
[Page 0] Raw (สส เขต):
ส.ส. ๕/๑๗

ชุดที่ ๑ (สีขาว) ส่งให้คณะกรรมการการเลือกตั้งประจำเขตเลือกตั้ง
ชุดที่ ๒ (สีเขียว) ปิดหน้าสถานที่นับคะแนนบัตรเลือกตั้ง
ชุดที่ ๓ (สีชมพู) ใส่ถุงวัสดุใสซึ่งบรรจุบัตรเลือกตั้งที่นับเป็นคะแนนแล้ว

รายงานผลการนับคะแนนบัตรเลือกตั้งนอกเขตเลือกตั้งและนอกราชอาณาจักรแบบแบ่งเขตเลือกตั้ง
ณ สถานที่นับคะแนน (ชื่อสถานที่) หอประชุมคาบสมรรถ เขตเลือกตั้งที่ 1 จังหวัด ศรีสะเกษ

คณะกรรมการนับคะแนนบัตรเลือกตั้งนอกเขตเลือกตั้งและนอกราชอาณาจักร ชุดที่ 1 เขตเลือกตั้ง ที่ 1 จังหวัด ศรีสะเกษ ได้รับของใส่บัตรเลือกตั้งจาก บริษัท ไปรษณีย์ไทย จำกัด หรือผู้แทน หน่วยงาน หรือองค์กรที่คณะกรรมการการเลือกตั้งมอบหมาย จำนวน 750 ซอง และได้ตรวจนับจำนวน บัตรเลือกตั้งในซองใส่บัตรเลือกตั้งแล้ว มีจำนวน 750 บัตร จึงได้ดำเนินการนับคะแนนบัตรเลือกตั้ง นอกเขตเลือกตั้งและนอกราชอาณาจักรแบบแบ่งเขตเลือกตั้งเสร็จเร

## Export DataFrame

In [19]:
df_ea_constituency = pd.DataFrame(ea_con_rows)

rename_map_ea = {f"ชื่อนามสกุลผู้สมัครหมายเลข{num}": CONSTITUENCY_CANDIDATES[num]
                 for num in range(1, 10)}
df_ea_con_display = df_ea_constituency.rename(columns=rename_map_ea)

print("=== df_ea_constituency (สส เขต ล่วงหน้านอกเขต) ===")
print(f"Rows: {len(df_ea_constituency)}")
display(df_ea_con_display)

=== df_ea_constituency (สส เขต ล่วงหน้านอกเขต) ===
Rows: 11


,source_pdf,ชุด,received_cards,good_cards,bad_cards,no_vote_cards,นายสุเกษม เรืองนุช,นายเนียม ผลบุญ,นายธเนส เครือรัตน์,นายวีรชัย จันทร์ดวงศรี,นายสิริพงศ์ อังคสกุลเกียรติ,นายสุรเดช นาจำปา,นายคณีศร คำโสภา,นายกุลธวัช สาลี,นางสาวฉวี ปิ่นทอม
0,data/ชุดที่1/1.pdf,ชุดที่1,750,700,15,3,312,8,94,10,251,4,9,7,9
1,data/ชุดที่2/2.pdf,ชุดที่2,750,707,8,35,299,9,138,3,242,4,3,3,6
2,data/ชุดที่3/3.pdf,ชุดที่3,750,707,10,33,307,8,115,3,256,4,8,1,5
3,data/ชุดที่4/4.pdf,ชุดที่4,350,705,15,30,372,8,16,5,200,5,5,0,0
4,data/ชุดที่5/5.pdf,ชุดที่5,750,701,13,36,310,10,112,6,243,2,8,3,7
5,data/ชุดที่6/6.pdf,ชุดที่6,750,716,6,28,714,8,107,6,260,9,9,6,9
6,data/ชุดที่7/7.pdf,ชุดที่7,750,710,7,33,329,13,101,7,229,5,12,8,10
7,data/ชุดที่8/8.pdf,ชุดที่8,750,696,17,35,318,8,97,12,240,4,5,6,8
8,data/ชุดที่9/9.pdf,ชุดที่9,750,706,8,36,319,8,103,3,238,6,13,4,12
9,data/ชุดที่10/10.pdf,ชุดที่10,750,710,10,26,309,7,104,9,256,4,8,5,11


In [20]:
df_ea_partylist = pd.DataFrame(ea_party_rows)

print("=== df_ea_partylist (บัญชีรายชื่อ ล่วงหน้านอกเขต) ===")
print(f"Rows: {len(df_ea_partylist)}")
display(df_ea_partylist)

=== df_ea_partylist (บัญชีรายชื่อ ล่วงหน้านอกเขต) ===
Rows: 11


,source_pdf,ชุด,received_cards,good_cards,bad_cards,no_vote_cards,ไทยทรัพย์ทวี,เพื่อชาติไทย,ใหม่,มิติใหม่,...,ไทยสร้างไทย,ไทยก้าวใหม่,ประชาอาสาชาติ,พร้อม,เครือข่ายชาวนาแห่งประเทศไทย,ไทยพิทักษ์ธรรม,ความหวังใหม่,ไทยรวมไทย,เพื่อบ้านเมือง,พลังไทยรักชาติ
0,data/ชุดที่1/1.pdf,ชุดที่1,790,790,0.0,16.0,0,0,0,0,...,5,2,0,1,0,0,0,0,0,0
1,data/ชุดที่2/2.pdf,ชุดที่2,750,756,4.0,20.0,0,0,0,0,...,1,5,0,1,0,0,0,0,0,0
2,data/ชุดที่3/3.pdf,ชุดที่3,750,728,7.0,15.0,0,0,0,0,...,5,2,0,0,0,0,0,0,0,1
3,data/ชุดที่4/4.pdf,ชุดที่4,350,330,5.0,14.0,0,2,0,0,...,3,1,0,0,0,0,0,0,0,0
4,data/ชุดที่5/5.pdf,ชุดที่5,731,9,10.0,NaN,0,6,3,0,...,8,2,0,4,0,0,0,0,0,0
5,data/ชุดที่6/6.pdf,ชุดที่6,750,720,7.0,23.0,1,9,1,0,...,5,1,0,0,0,0,0,0,1,1
6,data/ชุดที่7/7.pdf,ชุดที่7,750,726,4.0,20.0,1,7,0,0,...,3,5,0,0,0,0,0,6,0,0
7,data/ชุดที่8/8.pdf,ชุดที่8,750,924,NaN,15.0,0,0,0,0,...,3,2,0,0,0,0,0,1,0,0
8,data/ชุดที่9/9.pdf,ชุดที่9,750,715,6.0,29.0,0,0,0,0,...,3,3,0,0,0,0,0,0,0,0
9,data/ชุดที่10/10.pdf,ชุดที่10,750,720,16.0,14.0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [21]:
df_ea_debug = pd.DataFrame(ea_debug_rows)
print("=== debug ===")
display(df_ea_debug)

=== debug ===


,ชุด,source_pdf,con_pages_read,con_scores_count,con_completeness,party_pages_read,party_total_score,party_completeness
0,ชุดที่1,data/ชุดที่1/1.pdf,2,9,yes,4,725,COMPLETE
1,ชุดที่2,data/ชุดที่2/2.pdf,2,9,yes,4,728,COMPLETE
2,ชุดที่3,data/ชุดที่3/3.pdf,2,9,yes,4,734,COMPLETE
3,ชุดที่4,data/ชุดที่4/4.pdf,2,9,yes,4,511,COMPLETE
4,ชุดที่5,data/ชุดที่5/5.pdf,2,9,yes,4,749,COMPLETE
5,ชุดที่6,data/ชุดที่6/6.pdf,2,9,yes,4,637,COMPLETE
6,ชุดที่7,data/ชุดที่7/7.pdf,2,9,yes,4,319,COMPLETE
7,ชุดที่8,data/ชุดที่8/8.pdf,2,9,yes,4,724,COMPLETE
8,ชุดที่9,data/ชุดที่9/9.pdf,2,9,yes,4,715,COMPLETE
9,ชุดที่10,data/ชุดที่10/10.pdf,2,9,yes,4,720,COMPLETE


In [22]:
df_ea_con_display.to_csv("early_voting_constituency.csv",  index=False, encoding="utf-8-sig")
df_ea_partylist.to_csv("early_voting_partylist.csv",        index=False, encoding="utf-8-sig")
df_ea_debug.to_csv("early_voting_debug.csv",                index=False, encoding="utf-8-sig")

print("Saved: early_voting_constituency.csv")
print("Saved: early_voting_partylist.csv")
print("Saved: early_voting_debug.csv")

Saved: early_voting_constituency.csv
Saved: early_voting_partylist.csv
Saved: early_voting_debug.csv


In [23]:
from google.colab import files
files.download("early_voting_constituency.csv")
files.download("early_voting_partylist.csv")
files.download("early_voting_debug.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Pipeline: ล่วงหน้าในเขต

In [25]:
import os, zipfile
from google.colab import files
from pdf2image import convert_from_path
import typhoon_ocr
from bs4 import BeautifulSoup

uploaded = files.upload()

for fname, data in uploaded.items():
    with open(fname, 'wb') as f:
        f.write(data)

    pages = convert_from_path(fname, 300)
    img_path = "debug_page_0.jpg"
    pages[0].save(img_path, "JPEG")

    ocr_text = typhoon_ocr.ocr_document(img_path)
    print(ocr_text)
    print("="*60)

    soup = BeautifulSoup(ocr_text, "html.parser")
    for td in soup.find_all("td"):
        if "รวมคะแนนทั้งสิ้น" in td.get_text():
            tr = td.find_parent("tr")
            print("พบ 'รวมคะแนนทั้งสิ้น':", tr.get_text(separator=" | ", strip=True))

Saving ในเขต.pdf to ในเขต.pdf
ส.ส. ๕/๑๖ (บช)

<table><tr><td>หมายเลขของบัญชีรายชื่อ ของพรรคการเมือง</td><td>ชื่อพรรคการเมือง</td><td>ได้คะแนน ตัวเลข (อารบิก) / ตัวอักษร</td></tr><tr><td>53</td><td>ไทยพิทักษ์ธรรม</td><td>0 (ศูนย์)</td></tr><tr><td>54</td><td>ความหวังใหม่</td><td>0 (ศูนย์)</td></tr><tr><td>55</td><td>ไทยรวมไทย</td><td>0 (ศูนย์)</td></tr><tr><td>56</td><td>เพื่อบ้านเมือง</td><td>0 (ศูนย์)</td></tr><tr><td>57</td><td>พลังไทยรักชาติ</td><td>0 (ศูนย์)</td></tr><tr><td colspan="2">รวมคะแนนทั้งสิ้น</td><td>13 (ศูนย์)</td></tr></table>

ประกาศ ณ วันที่ 8 เดือน กุมภาพันธ์ พ.ศ. 2569

(ลงชื่อ)  ประธานกรรมการนับคะแนนบัตรเลือกตั้ง
(น.ส.ชัยชลลี สำภา)

(ลงชื่อ)  กรรมการนับคะแนนบัตรเลือกตั้ง
(น.ส.ปิยาภรณ์ ส่งหันชม)

(ลงชื่อ)  กรรมการนับคะแนนบัตรเลือกตั้ง
(นายเดชณรงค์ คำสว่าง)

(ลงชื่อ)  กรรมการนับคะแนนบัตรเลือกตั้ง
(น.ส.พญรี ศรีสมุทร)

(ลงชื่อ)  กรรมการนับคะแนนบัตรเลือกตั้ง
(นางอาญา สุมาลี)

(ลงชื่อ)  กรรมการนับคะแนนบัตรเลือกตั้ง
(น.ส.ศิวาพร โพลพิมพ์)

(ลงชื่อ)  กรรมการนับคะแนนบัตรเลือ